In [0]:
# ==============================================================================
# Pipeline Step: 03_incremental_trips_merge.py
# Description: Ingests incremental trip data (e.g., Day 2 batch), applies schema
#              standardization and deduplication, and appends the new records
#              into the Silver trips Delta table.
# ==============================================================================

from pyspark.sql.functions import col, current_timestamp, to_timestamp
from pyspark.sql.types import IntegerType

# ------------------------------------------------------------------------------
# 1. Storage Account Credentials & Container Paths
# ------------------------------------------------------------------------------
storage_account = "sttransitanalyticsdev"
storage_key = "sssshtfcxxxx
# Define ADLS Gen2 ABFSS endpoints
BRONZE_PATH = f"abfss://bronze@{storage_account}.dfs.core.windows.net"
SILVER_PATH = f"abfss://silver@{storage_account}.dfs.core.windows.net"

# Store storage credentials configuration
storage_options = {
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net": storage_key
}

# ------------------------------------------------------------------------------
# 2. Ingest & Transform Incremental Batch Data (Day 2 Trips)
# ------------------------------------------------------------------------------
df_new_trips = (
    spark.read.options(**storage_options)
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(f"{BRONZE_PATH}/trips_02.csv")
    .dropDuplicates(["trip_id"])
    .withColumn(
        "start_time",
        to_timestamp(col("start_time"), "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn(
        "end_time",
        to_timestamp(col("end_time"), "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn("passenger_count", col("passenger_count").cast(IntegerType()))
    .withColumn("ingested_at", current_timestamp())
)

# ------------------------------------------------------------------------------
# 3. Append Incremental Batch into Silver Delta Table
# ------------------------------------------------------------------------------
(
    df_new_trips.write.options(**storage_options)
    .format("delta")
    .mode("append")
    .save(f"{SILVER_PATH}/trips")
)

print("⚡ INCREMENTAL LOAD SUCCESS: Day 2 trips appended into Silver Delta Lake!")

⚡ INCREMENTAL LOAD SUCCESS: Day 2 trips appended into Silver Delta Lake!
